In [1]:
import re
import json
import pandas as pd


RUNTEXT_PATH = "../srd2014/RUNTEXT"
ENGLISH_PATH = "../srd2014/ENGLISH"

In [10]:
def split_signum_and_text(line):
    """
    Разделяет строку на:
    signum
    text
    """
    tokens = line.strip().split()

    if len(tokens) < 3:
        return None, None

    signum_parts = []
    text_start_index = 0

    for i, token in enumerate(tokens):
        # первые два токена всегда часть signum (Öl 1, U 11 и т.п.)
        if i < 2:
            signum_parts.append(token)
            continue

        # маркеры
        if token in {"$", "†", "M", "U", "SENTIDA"}:
            signum_parts.append(token)
            continue

        # §A, §P и т.д.
        if token.startswith("§"):
            signum_parts.append(token)
            continue

        # если не маркер — значит начинается текст
        text_start_index = i
        break

    signum = " ".join(signum_parts)
    text = " ".join(tokens[text_start_index:])

    return signum, text


def load_parallel(runtxt_path, eng_path):
    data = []

    with open(runtxt_path, "r", encoding="utf-8") as f1, \
         open(eng_path, "r", encoding="utf-8") as f2:

        for l1, l2 in zip(f1, f2):

            s1, t1 = split_signum_and_text(l1)
            s2, t2 = split_signum_and_text(l2)

            if not s1 or not s2:
                continue

            # проверяем совпадение signum
            if s1 != s2:
                continue

            # фильтрация
            if "SENTIDA" in s1:
                continue
            if not t1.strip() or not t2.strip():
                continue

            data.append({
                "signum": s1,
                "transliteration": t1.strip(),
                "english": t2.strip()
            })

    return data


def save_csv(data, path="runic_parallel.csv"):
    df = pd.DataFrame(data)
    df.to_csv(path, index=False)
    print("Saved:", path, "Pairs:", len(df))


def save_jsonl(data, path="runic_finetune.jsonl"):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            json.dump({
                "messages": [
                    {"role": "system", "content": "Translate Old Norse runic transliteration into English."},
                    {"role": "user", "content": row["transliteration"]},
                    {"role": "assistant", "content": row["english"]}
                ]
            }, f)
            f.write("\n")
    print("Saved:", path)



In [13]:
dataset = load_parallel(RUNTEXT_PATH, ENGLISH_PATH)
print("Total pairs:", len(dataset))

save_csv(dataset, "srd_parallel.csv")

Total pairs: 11601
Saved: srd_parallel.csv Pairs: 11601
